In [17]:
import pandas as pd

data = pd.read_csv("data.csv", sep=';')

print(data.shape)
print(data.columns)
print(data.head())

(4424, 37)
Index(['Marital status', 'Application mode', 'Application order', 'Course',
       'Daytime/evening attendance\t', 'Previous qualification',
       'Previous qualification (grade)', 'Nacionality',
       'Mother's qualification', 'Father's qualification',
       'Mother's occupation', 'Father's occupation', 'Admission grade',
       'Displaced', 'Educational special needs', 'Debtor',
       'Tuition fees up to date', 'Gender', 'Scholarship holder',
       'Age at enrollment', 'International',
       'Curricular units 1st sem (credited)',
       'Curricular units 1st sem (enrolled)',
       'Curricular units 1st sem (evaluations)',
       'Curricular units 1st sem (approved)',
       'Curricular units 1st sem (grade)',
       'Curricular units 1st sem (without evaluations)',
       'Curricular units 2nd sem (credited)',
       'Curricular units 2nd sem (enrolled)',
       'Curricular units 2nd sem (evaluations)',
       'Curricular units 2nd sem (approved)',
       'Curricula

In [18]:
# Remove 'Enrolled' (optional but better)
data = data[data['Target'] != 'Enrolled']

# Convert to binary
data['dropout'] = data['Target'].apply(lambda x: 1 if x == 'Dropout' else 0)

print(data['dropout'].value_counts())

dropout
0    2209
1    1421
Name: count, dtype: int64


In [19]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from imblearn.over_sampling import SMOTE
import pickle

# Load dataset
data = pd.read_csv("data.csv", sep=';')

# Remove 'Enrolled'
data = data[data['Target'] != 'Enrolled']

# Convert to binary
data['dropout'] = data['Target'].apply(lambda x: 1 if x == 'Dropout' else 0)

# Features
feature_cols = [
    'Curricular units 1st sem (grade)',
    'Curricular units 2nd sem (grade)',
    'Age at enrollment',
    'Debtor'
]

X = data[feature_cols]
y = data['dropout']

# Balance dataset
smote = SMOTE(random_state=42)
X_res, y_res = smote.fit_resample(X, y)

print("Balanced:\n", pd.Series(y_res).value_counts())

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X_res, y_res, test_size=0.2, random_state=42
)

# 🔥 CHANGE 1: Improved model (reduce overconfidence)
model = RandomForestClassifier(
    n_estimators=200,
    max_depth=5,
    random_state=42
)

model.fit(X_train, y_train)

# Accuracy
print("Accuracy:", model.score(X_test, y_test))

# 🔥 CHANGE 2: Save feature names also
pickle.dump((model, feature_cols), open("model.pkl", "wb"))

print("✅ Model trained and saved!")

Balanced:
 dropout
1    2209
0    2209
Name: count, dtype: int64
Accuracy: 0.8495475113122172
✅ Model trained and saved!


In [20]:
!pip install streamlit pyngrok imbalanced-learn

In [21]:
import pandas as pd
import pickle

# Load model
model, feature_cols = pickle.load(open("model.pkl", "rb"))

# Final test cases
test_cases = [
    [18, 17, 20, 0],   # 1. Excellent student
    [15, 14, 19, 0],   # 2. Good student
    [10, 10, 21, 0],   # 3. Average student
    [8, 7, 23, 0],     # 4. Below average
    [5, 6, 22, 1],     # 5. Weak + debtor
    [2, 3, 30, 1],     # 6. Very poor
    [12, 11, 28, 1],   # 7. Medium + debtor
    [16, 15, 35, 0],   # 8. Good but older
]

print("===== FINAL MODEL TEST =====\n")

for i, case in enumerate(test_cases, 1):
    df = pd.DataFrame([case], columns=feature_cols)

    pred = model.predict(df)
    prob = model.predict_proba(df)

    risk = prob[0][1] * 100

    print(f"Test Case {i}: {case}")
    print(f"Prediction: {'Dropout' if pred[0]==1 else 'No Dropout'}")
    print(f"Risk: {risk:.2f}%")
    print("-"*40)

===== FINAL MODEL TEST =====

Test Case 1: [18, 17, 20, 0]
Prediction: No Dropout
Risk: 16.16%
----------------------------------------
Test Case 2: [15, 14, 19, 0]
Prediction: No Dropout
Risk: 11.71%
----------------------------------------
Test Case 3: [10, 10, 21, 0]
Prediction: Dropout
Risk: 94.11%
----------------------------------------
Test Case 4: [8, 7, 23, 0]
Prediction: Dropout
Risk: 96.46%
----------------------------------------
Test Case 5: [5, 6, 22, 1]
Prediction: Dropout
Risk: 94.82%
----------------------------------------
Test Case 6: [2, 3, 30, 1]
Prediction: Dropout
Risk: 99.39%
----------------------------------------
Test Case 7: [12, 11, 28, 1]
Prediction: Dropout
Risk: 85.60%
----------------------------------------
Test Case 8: [16, 15, 35, 0]
Prediction: No Dropout
Risk: 27.94%
----------------------------------------
